# UBER SUPPLY-DEMAND GAP ANALYSIS - EXPLORATORY DATA ANALYSIS (EDA)

### 1. Connecting to PostgreSQL

In [1]:
# Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot
import seaborn as sns
import psycopg2
from sqlalchemy import create_engine

In [2]:
# Connection configuration
db_config = {
    'host': 'localhost',
    'database': 'uber',
    'user': 'postgres',
    'password': 'root' # your actual password
}

connection_string = f"postgresql://{db_config['user']}:{db_config['password']}@{db_config['host']}/{db_config['database']}"

try:
    # Create SQLAlchemy engine
    engine = create_engine(connection_string)
    
    # Test connection
    test_query = "SELECT version();"
    result = pd.read_sql(test_query, engine)
    
    print("Database connection successful!")
    print(f"Connected to: {db_config['database']} database")
    print(f"PostgreSQL Version: {result.iloc[0,0][:50]}...")
    
except Exception as e:
    print(f"Connection failed: {e}")

Database connection successful!
Connected to: uber database
PostgreSQL Version: PostgreSQL 17.5 on x86_64-windows, compiled by msv...


In [3]:
# Importing data from the database
table = 'uber_requests'
query = f"SELECT * FROM {table}"
df = pd.read_sql(query, engine)
print(f"{table}: {df.shape[0]} rows, {df.shape[1]} columns.")

uber_requests: 6745 rows, 10 columns.


In [4]:
df

,request_id,pickup_point,driver_id,status,request_timestamp,drop_timestamp,request_timestamp_clean,drop_timestamp_clean,request_hour,time_slot
0,380,Airport,8,Trip Completed,11/7/2016 8:18,11/7/2016 9:18,2016-11-07 08:18:00,2016-11-07 09:18:00,8,Morning
1,1050,Airport,8,Trip Completed,11/7/2016 19:39,11/7/2016 20:30,2016-11-07 19:39:00,2016-11-07 20:30:00,19,Evening
2,1769,City,8,Trip Completed,12/7/2016 8:57,12/7/2016 9:24,2016-12-07 08:57:00,2016-12-07 09:24:00,8,Morning
3,2520,City,8,Trip Completed,12/7/2016 21:05,12/7/2016 22:20,2016-12-07 21:05:00,2016-12-07 22:20:00,21,Late Evening
4,3265,Airport,8,Trip Completed,13-07-2016 10:22:07,13-07-2016 11:07:02,2016-07-13 10:22:07,2016-07-13 11:07:02,10,Morning
...,...,...,...,...,...,...,...,...,...,...
6740,6722,Airport,NA,No Cars Available,15-07-2016 23:18:21,NA,2016-07-15 23:18:21,NaT,23,Late Night
6741,6725,Airport,NA,No Cars Available,15-07-2016 23:21:53,NA,2016-07-15 23:21:53,NaT,23,Late Night
6742,6728,City,NA,No Cars Available,15-07-2016 23:26:50,NA,2016-07-15 23:26:50,NaT,23,Late Night
6743,6730,Airport,NA,No Cars Available,15-07-2016 23:27:55,NA,2016-07-15 23:27:55,NaT,23,Late Night


### 2. Data Validation

In [5]:
# Basic Validaton:
# shape
df.shape

(6745, 10)

In [6]:
# column
df.columns

Index(['request_id', 'pickup_point', 'driver_id', 'status',
       'request_timestamp', 'drop_timestamp', 'request_timestamp_clean',
       'drop_timestamp_clean', 'request_hour', 'time_slot'],
      dtype='object')

In [7]:
# data types
df.dtypes

request_id                          int64
pickup_point                       object
driver_id                          object
status                             object
request_timestamp                  object
drop_timestamp                     object
request_timestamp_clean    datetime64[ns]
drop_timestamp_clean       datetime64[ns]
request_hour                        int64
time_slot                          object
dtype: object

In [8]:
# missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6745 entries, 0 to 6744
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   request_id               6745 non-null   int64         
 1   pickup_point             6745 non-null   object        
 2   driver_id                6745 non-null   object        
 3   status                   6745 non-null   object        
 4   request_timestamp        6745 non-null   object        
 5   drop_timestamp           6745 non-null   object        
 6   request_timestamp_clean  6745 non-null   datetime64[ns]
 7   drop_timestamp_clean     2831 non-null   datetime64[ns]
 8   request_hour             6745 non-null   int64         
 9   time_slot                6745 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(6)
memory usage: 527.1+ KB


In [9]:
df.isnull().sum()

request_id                    0
pickup_point                  0
driver_id                     0
status                        0
request_timestamp             0
drop_timestamp                0
request_timestamp_clean       0
drop_timestamp_clean       3914
request_hour                  0
time_slot                     0
dtype: int64

In [10]:
# unique values:
df.nunique()

request_id                 6745
pickup_point                  2
driver_id                   301
status                        3
request_timestamp          5618
drop_timestamp             2599
request_timestamp_clean    5618
drop_timestamp_clean       2598
request_hour                 24
time_slot                     7
dtype: int64

### Data Validation Report

#### Data Validation Report

##### Executive Summary
**Data Quality: EXCELLENT** - All validation checks passed  
**Dataset Ready**: 6,745 records loaded successfully from PostgreSQL  

##### Basic Checks
- **Row Count**: 6,745 records
- **Column Count**: 10 columns
- **Unique IDs**: All request_ids unique
- **Data Types**: Correct (datetime, int64, object)
- **Missing Values**: Only expected nulls in drop_timestamp

##### Conclusion
Dataset validated and ready for exploratory data analysis.


### 3. Exploratory Data Analysis

In [11]:
# 1. Summary Statistics
df.describe()

,request_id,request_timestamp_clean,drop_timestamp_clean,request_hour
count,6745.000000,6745,2831,6745.000000
mean,3384.644922,2016-09-04 08:49:18.446849536,2016-09-05 17:03:26.523843072,12.956709
min,1.000000,2016-07-13 00:00:24,2016-07-13 00:01:12,0.000000
25%,1691.000000,2016-07-14 07:38:26,2016-07-14 08:10:24.500000,7.000000
50%,3387.000000,2016-07-15 13:44:15,2016-07-15 15:27:41,13.000000
75%,5080.000000,2016-11-07 19:00:00,2016-11-07 19:25:30,19.000000
max,6766.000000,2016-12-07 23:54:00,2016-12-07 23:45:00,23.000000
std,1955.099667,NaN,NaN,6.504052


In [12]:
# Key Insights from Summary Statistics:
# - Request IDs well-distributed (1-6,766) indicating consistent data collection
# - Data spans July-December 2016 with concentration in middle period (Sept median)  
# - Hourly demand centered around 1 PM with 6.5 hour spread across day
# - Evening rush evident (75th percentile at 7 PM) and morning activity at 7 AM
# - Only 42% trips completed (2,831/6,745) confirming high failure rate

In [13]:
# request_hour distribution:
df['request_hour'].value_counts().sort_index()

request_hour
0      99
1      85
2      99
3      92
4     203
5     445
6     398
7     406
8     423
9     431
10    243
11    171
12    184
13    160
14    136
15    171
16    159
17    418
18    510
19    473
20    492
21    449
22    304
23    194
Name: count, dtype: int64

In [14]:
# Key Insights from Hourly Distribution:
# - Peak demand occurs at 6 PM (510 requests) with strong evening rush from 5-8 PM (1,401 requests)
# - Early morning surge at 5-7 AM (1,249 requests) indicates commuter travel patterns  
# - Lowest demand during late night hours 0-3 AM (375 requests) showing minimal overnight activity

In [15]:
# request_hour statistics:
print(f"Min hour: {df['request_hour'].min()}")
print(f"Max hour: {df['request_hour'].max()}")
print(f"Most common hour: {df['request_hour'].mode()[0]}")
print(f"Least common hour: {df['request_hour'].value_counts().idxmin()}")

Min hour: 0
Max hour: 23
Most common hour: 18
Least common hour: 1


In [16]:
# Key Insights from Hourly Statistics:
# - Peak demand at 6 PM (hour 18) indicates evening rush as highest demand period
# - Lowest demand at 1 AM (hour 1) confirms minimal overnight service requirements

In [17]:
# 2. Categorical variables analysis:
# status distribution:
status_counts = df['status'].value_counts()
status_counts

status
Trip Completed       2831
No Cars Available    2650
Cancelled            1264
Name: count, dtype: int64

In [18]:
status_pct = df['status'].value_counts(normalize=True) * 100
status_pct

status
Trip Completed       41.971831
No Cars Available    39.288362
Cancelled            18.739807
Name: proportion, dtype: float64

In [19]:
for status, count in status_counts.items():
   pct = status_pct[status]
   print(f"{status}: {count:,} ({pct:.1f}%)")

Trip Completed: 2,831 (42.0%)
No Cars Available: 2,650 (39.3%)
Cancelled: 1,264 (18.7%)


In [20]:
print(f"Total Requests: {status_counts.sum()}")

Total Requests: 6745


In [21]:
# pickup_point distribution:
pickup_counts = df['pickup_point'].value_counts()
pickup_counts

pickup_point
City       3507
Airport    3238
Name: count, dtype: int64

In [22]:
pickup_pct = df['pickup_point'].value_counts(normalize=True) * 100
pickup_pct

pickup_point
City       51.99407
Airport    48.00593
Name: proportion, dtype: float64

In [23]:
for pickup, count in pickup_counts.items():
   pct = pickup_pct[pickup]
   print(f"{pickup}: {count:,} ({pct:.1f}%)")

City: 3,507 (52.0%)
Airport: 3,238 (48.0%)


In [24]:
# time_slot distribution:
timeslot_counts = df['time_slot'].value_counts()
timeslot_counts

time_slot
Evening          1560
Early Morning    1452
Morning          1268
Late Evening      941
Afternoon         651
Late Night        498
Night             375
Name: count, dtype: int64

In [25]:
timeslot_pct = df['time_slot'].value_counts(normalize=True) * 100
timeslot_pct

time_slot
Evening          23.128243
Early Morning    21.527057
Morning          18.799110
Late Evening     13.951075
Afternoon         9.651594
Late Night        7.383247
Night             5.559674
Name: proportion, dtype: float64

In [26]:
for slot, count in timeslot_counts.items():
    pct = timeslot_pct[slot]
    print(f"{slot}: {count:,} ({pct:.1f}%)")

Evening: 1,560 (23.1%)
Early Morning: 1,452 (21.5%)
Morning: 1,268 (18.8%)
Late Evening: 941 (14.0%)
Afternoon: 651 (9.7%)
Late Night: 498 (7.4%)
Night: 375 (5.6%)


In [27]:
# Key Insights from Categorical Analysis:
# - Only 42% trips complete successfully while 58% fail (39% no cars, 19% cancelled)
# - City pickups slightly dominate (52%) over airport pickups (48%) showing balanced coverage
# - Evening period has highest demand (23%) followed by early morning rush (21.5%)
# - Combined rush hours (evening + early morning) account for 45% of all requests
# - Night operations minimal (13% combined late night + night) indicating reduced overnight service

In [28]:
# 3. Cross-Tablulation:
# status by pickup_point:
status_pickup = df.groupby(['status', 'pickup_point']).size().unstack()
status_pickup

pickup_point,Airport,City
status,,
Cancelled,198,1066
No Cars Available,1713,937
Trip Completed,1327,1504


In [29]:
status_pickup_pct = df.groupby(['pickup_point', 'status']).size().unstack()
status_pickup_pct = status_pickup_pct.div(status_pickup_pct.sum(axis=1), axis=0).round(2) * 100
status_pickup_pct

status,Cancelled,No Cars Available,Trip Completed
pickup_point,,,
Airport,6.0,53.0,41.0
City,30.0,27.0,43.0


In [30]:
# status by time_slot:
status_time = df.groupby(['time_slot', 'status']).size().unstack()
status_time

status,Cancelled,No Cars Available,Trip Completed
time_slot,,,
Afternoon,69,182,400
Early Morning,541,307,604
Evening,105,883,572
Late Evening,83,555,303
Late Night,22,219,257
Morning,430,279,559
Night,14,225,136


In [31]:
status_time_pct = df.groupby(['time_slot', 'status']).size().unstack()
status_time_pct = status_time_pct.div(status_time_pct.sum(axis=1), axis=0).round(2) * 100
status_time_pct

status,Cancelled,No Cars Available,Trip Completed
time_slot,,,
Afternoon,11.0,28.0,61.0
Early Morning,37.0,21.0,42.0
Evening,7.0,57.0,37.0
Late Evening,9.0,59.0,32.0
Late Night,4.0,44.0,52.0
Morning,34.0,22.0,44.0
Night,4.0,60.0,36.0


In [32]:
# Key Insights from Cross-Tabulation Analysis:
# - Airport has major supply shortage (53% no cars available) vs City (27% no cars)
# - City has higher cancellation problem (30% cancelled) vs Airport (6% cancelled) 
# - Evening period dominated by "No Cars Available" (57%) indicating peak supply crisis
# - Early Morning has highest cancellation rate (37%) suggesting driver reluctance
# - Night period shows 60% no cars available confirming overnight supply shortage
# - Only Morning and Afternoon achieve reasonable completion rates (44% and 61%)

In [33]:
# 4. Temporal Analysis
# Peak hours:
top_hours = df['request_hour'].value_counts().head()
top_hours

request_hour
18    510
20    492
19    473
21    449
5     445
Name: count, dtype: int64

In [34]:
# Quiet hours:
quiet_hours = df['request_hour'].value_counts().tail()
quiet_hours

request_hour
14    136
0      99
2      99
3      92
1      85
Name: count, dtype: int64

In [35]:
# time_slot performance analysis
# Success Rate by time_slot:
success_by_time = df.groupby('time_slot')['status'].apply(lambda x: (x == 'Trip Completed').mean() * 100).round(1)
success_by_time.sort_values(ascending=False)

time_slot
Afternoon        61.4
Late Night       51.6
Morning          44.1
Early Morning    41.6
Evening          36.7
Night            36.3
Late Evening     32.2
Name: status, dtype: float64

In [36]:
# Failure Rate by time_slot:
failure_by_time = df.groupby('time_slot')['status'].apply(lambda x: (x != "Trip Completed").mean() * 100).round(1)
failure_by_time.sort_values(ascending=False)


time_slot
Late Evening     67.8
Night            63.7
Evening          63.3
Early Morning    58.4
Morning          55.9
Late Night       48.4
Afternoon        38.6
Name: status, dtype: float64

In [37]:
# Success Rate by hour:
success_by_hour = df.groupby('request_hour')['status'].apply(lambda x: (x == 'Trip Completed').mean() * 100).round(1)
success_by_hour

request_hour
0     40.4
1     29.4
2     37.4
3     37.0
4     38.4
5     41.6
6     42.0
7     42.9
8     36.6
9     40.1
10    47.7
11    67.3
12    65.8
13    55.6
14    64.7
15    59.6
16    57.2
17    36.1
18    32.2
19    35.1
20    32.7
21    31.6
22    50.7
23    53.1
Name: status, dtype: float64

In [38]:
# Key Insights from Temporal Analysis:
# - Afternoon achieves best success rate (61.4%) while late evening worst (32.2%)
# - Late evening has highest failure rate (67.8%) indicating peak demand vs supply mismatch
# - Hour 11 AM shows exceptional performance (67.3% success) - optimal supply-demand balance
# - Evening rush hours 5-8 PM consistently underperform (32-36% success rates)
# - Night hours 0-3 AM show moderate performance (29-40%) despite low volume

In [39]:
# 5. Supply Demand Gap Analysis
# Calculate supply-demand gaps by location and time
gap_analysis = df.groupby(['pickup_point', 'time_slot', 'status']).size().unstack(fill_value=0)
gap_analysis

status                      Cancelled  No Cars Available  Trip Completed
pickup_point time_slot                                                  
Airport      Afternoon             36                 55             187
             Early Morning         15                 44             277
             Evening               63                801             276
             Late Evening          57                529             135
             Late Night             3                136             142
             Morning               24                 34             243
             Night                  0                114              67
City         Afternoon             33                127             213
             Early Morning        526                263             327
             Evening               42                 82             296
             Late Evening          26                 26             168
             Late Night            19                 83             115
             Morning              406                245             316
             Night                 14                111              69

In [40]:
gap_analysis['Total_Requests'] = gap_analysis.sum(axis=1)

In [41]:
gap_analysis['Failed_Requests'] = gap_analysis['Cancelled'] + gap_analysis['No Cars Available']

In [42]:
gap_analysis['Gap_Percentage'] = (gap_analysis['Failed_Requests'] / gap_analysis['Total_Requests'] * 100).round(1)

In [43]:
gap_analysis[['Total_Requests', 'Failed_Requests', 'Gap_Percentage']].sort_values('Gap_Percentage', ascending=False)

status                      Total_Requests  Failed_Requests  Gap_Percentage
pickup_point time_slot                                                     
Airport      Late Evening              721              586            81.3
             Evening                  1140              864            75.8
City         Early Morning            1116              789            70.7
             Morning                   967              651            67.3
             Night                     194              125            64.4
Airport      Night                     181              114            63.0
             Late Night                281              139            49.5
City         Late Night                217              102            47.0
             Afternoon                 373              160            42.9
Airport      Afternoon                 278               91            32.7
City         Evening                   420              124            29.5
             Late Evening              220               52            23.6
Airport      Morning                   301               58            19.3
             Early Morning             336               59            17.6

In [44]:
# Identify worst supply-demand scenarios
worst_gaps = gap_analysis.sort_values('Gap_Percentage', ascending=False).head()
worst_gaps[['Total_Requests', 'Failed_Requests', 'Gap_Percentage']]

status                      Total_Requests  Failed_Requests  Gap_Percentage
pickup_point time_slot                                                     
Airport      Late Evening              721              586            81.3
             Evening                  1140              864            75.8
City         Early Morning            1116              789            70.7
             Morning                   967              651            67.3
             Night                     194              125            64.4

In [45]:
# Identify worst supply-demand scenarios
best_performance = gap_analysis.sort_values('Gap_Percentage', ascending=True).head()
best_performance[['Total_Requests', 'Failed_Requests', 'Gap_Percentage']]

status                      Total_Requests  Failed_Requests  Gap_Percentage
pickup_point time_slot                                                     
Airport      Early Morning             336               59            17.6
             Morning                   301               58            19.3
City         Late Evening              220               52            23.6
             Evening                   420              124            29.5
Airport      Afternoon                 278               91            32.7

In [46]:
# Key Insights from Supply-Demand Gap Analysis:
# - Airport late evening shows worst gap (81.3% failure) with 586 failed out of 721 requests
# - Airport evening rush critically underserved (75.8% failure) - highest volume crisis (1,140 requests)
# - City early morning has severe driver shortage (70.7% failure) affecting 789 out of 1,116 requests
# - Airport early morning performs best (17.6% failure) indicating optimal supply-demand balance
# - Combined airport evening periods (evening + late evening) account for 1,861 requests with 76% failure rate


In [47]:
# 6. Driver Utilization Analysis
# Driver availability analysis
print(f"Total unique drivers: {df[df['driver_id'] != 'NA']['driver_id'].nunique()}")
print(f"Records with 'NA' drivers: {(df['driver_id'] == 'NA').sum()}")
print(f"Records with assigned drivers: {(df['driver_id'] != 'NA').sum()}")


Total unique drivers: 300
Records with 'NA' drivers: 2650
Records with assigned drivers: 4095


In [48]:
# Top 10 most active drivers
active_drivers = df[df['driver_id'] != 'NA']['driver_id'].value_counts()
active_drivers.head(10)

driver_id
27     22
22     21
84     21
70     21
177    21
176    21
197    20
69     20
114    20
24     20
Name: count, dtype: int64

In [49]:
# Driver utilisation by status
driver_status = df[df['driver_id'] != 'NA'].groupby('status').size()
driver_status

status
Cancelled         1264
Trip Completed    2831
dtype: int64

In [50]:
# Driver availability by time_slot:
driver_time = df.groupby('time_slot')['driver_id'].apply(lambda x: (x != 'NA').sum())
driver_time.sort_values(ascending=False)

time_slot
Early Morning    1145
Morning           989
Evening           677
Afternoon         469
Late Evening      386
Late Night        279
Night             150
Name: driver_id, dtype: int64

In [51]:
# Key Insights from Driver Utilization Analysis:
# - 300 active drivers handle 4,095 assigned requests while 2,650 requests get no driver assignment
# - Most active drivers handle 20-22 trips each showing relatively balanced workload distribution
# - Early morning has highest driver availability (1,145 assignments) despite high cancellation rates
# - Night period shows severe driver shortage (only 150 assignments) explaining supply gaps
# - Driver cancellation rate is 31% (1,264 cancelled out of 4,095 assigned) indicating driver reluctance issues

### Exploratory Data Analysis Report

#### Uber Supply-Demand Gap Analysis - EDA Report

##### Executive Summary
**Analysis Scope**: 6,745 Uber ride requests from July 2016  
**Key Finding**: Only 42% trip completion rate with critical supply-demand gaps  
**Status**: Comprehensive patterns identified across temporal and geographic dimensions  

##### Summary Statistics Analysis
- **Request distribution**: Well-distributed IDs (1-6,766) with consistent data collection
- **Temporal coverage**: July-December 2016 with September median concentration  
- **Hourly demand**: Centered around 1 PM with evening rush at 7 PM
- **Peak identification**: 75th percentile at hour 19 showing evening concentration
- **Completion rate**: Only 42% trips completed confirming high failure rate

##### Categorical Variables Analysis
- **Trip outcomes**: 42% completed, 39% no cars available, 19% cancelled
- **Geographic split**: City pickups dominate slightly (52%) over airport (48%)
- **Peak periods**: Evening (23%) and early morning (21.5%) account for 45% of requests
- **Rush hours**: Combined peak periods represent nearly half of all demand
- **Night operations**: Minimal activity (13%) indicating reduced overnight service

##### Cross-tabulation Analysis
- **Airport supply crisis**: 53% no cars available vs 27% in city
- **City cancellation issue**: 30% cancelled vs 6% at airport
- **Evening supply shortage**: 57% no cars available during peak demand
- **Morning driver reluctance**: 37% cancellation rate in early morning
- **Night availability**: 60% no cars confirming overnight supply gaps
- **Best performance**: Only afternoon (61%) and morning (44%) achieve reasonable rates

##### Temporal Analysis
- **Best performance**: Afternoon achieves 61.4% success rate
- **Worst performance**: Late evening shows 67.8% failure rate
- **Optimal hour**: 11 AM demonstrates 67.3% success rate
- **Rush hour crisis**: Evening hours 5-8 PM consistently underperform (32-36%)
- **Night patterns**: Moderate performance (29-40%) despite low volume

##### Supply-Demand Gap Analysis
- **Worst scenario**: Airport late evening (81.3% failure, 586 failed requests)
- **Highest volume crisis**: Airport evening rush (75.8% failure, 1,140 requests)
- **City morning problem**: 70.7% failure affecting 789 requests
- **Best scenario**: Airport early morning (17.6% failure rate)
- **Critical periods**: Combined airport evening periods show 76% failure rate

##### Driver Utilization Analysis
- **Active drivers**: 300 drivers handle 4,095 requests, 2,650 get no assignment
- **Workload balance**: Most active drivers handle 20-22 trips each
- **Peak availability**: Early morning shows 1,145 driver assignments
- **Night shortage**: Only 150 night assignments explaining supply gaps
- **Cancellation rate**: 31% of assigned rides cancelled indicating driver issues

##### Business Impact Summary
**Primary Crisis Points**: Airport evening periods and city morning cancellations  
**Supply Shortage**: Night operations critically understaffed  
**Revenue Loss**: 58% failed requests representing significant opportunity cost 